In [53]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [54]:
# Load the data

data = pd.read_csv("housing.csv")
data.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [55]:
# Creating an extra column for further splitting

data['income_cat'] = pd.cut(data['median_income'], bins=[0.0,1.5,3.0,4.5,6.0,np.inf], labels=[1,2,3,4,5])
data.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_cat
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,5
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,5
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,5
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,4
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,3


In [56]:
# Split the data in Train and Test set.
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(data, data["income_cat"]):
    strat_train_set = data.loc[train_index].drop("income_cat", axis = 1)  # Dropping the income_cat column as it is not needed anymore.
    strat_test_set = data.loc[test_index].drop("income_cat", axis = 1)    

In [57]:
data = strat_train_set.copy()

In [58]:
data_labels = data["median_house_value"].copy()
data = data.drop("median_house_value", axis = 1)

In [59]:
num_attributes= data.drop("ocean_proximity", axis =1).columns.tolist()
cat_attributes = ["ocean_proximity"]

In [60]:
# Make Pipelines

num_pipeline = Pipeline([
    ("imputer" , SimpleImputer(strategy= "most_frequent")),
    ("standard", StandardScaler())
])

cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown= "ignore"))
])

In [61]:
# Final Pipeline

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attributes),
    ("cat", cat_pipeline, cat_attributes),
])

In [62]:
data_prepared = full_pipeline.fit_transform(data)
print(data_prepared)

[[-0.94135046  1.34743822  0.02756357 ...  0.          0.
   0.        ]
 [ 1.17178212 -1.19243966 -1.72201763 ...  0.          0.
   1.        ]
 [ 0.26758118 -0.1259716   1.22045984 ...  0.          0.
   0.        ]
 ...
 [-1.5707942   1.31001828  1.53856552 ...  0.          0.
   0.        ]
 [-1.56080303  1.2492109  -1.1653327  ...  0.          0.
   0.        ]
 [-1.28105026  2.02567448 -0.13148926 ...  0.          0.
   0.        ]]


In [63]:
# Training data by Linear Regression

lin_reg = LinearRegression()
lin_reg.fit(data_prepared, data_labels)
lin_pred = lin_reg.predict(data_prepared)
lin_rmses = -cross_val_score(lin_reg,data_prepared, data_labels, scoring="neg_root_mean_squared_error", cv=10)
print(pd.Series(lin_rmses).describe())

count       10.000000
mean     69215.877312
std       2510.788398
min      65294.682372
25%      67134.217947
50%      69382.867759
75%      70730.992905
max      73028.626449
dtype: float64


In [ ]:
# Training data by Decision Tree

dec_reg = DecisionTreeRegressor()
dec_reg.fit(data_prepared, data_labels)
dec_pred = dec_reg.predict(data_prepared)
dec_rmses = -cross_val_score(dec_reg,data_prepared, data_labels, scoring="neg_root_mean_squared_error", cv=10)
print(pd.Series(dec_rmses).describe())

count       10.000000
mean     68952.963220
std       1423.367576
min      67325.491437
25%      67791.969857
50%      68747.850396
75%      69819.878353
max      71460.996623
dtype: float64


In [66]:
# Training data by Random Forest

random_forest_reg = RandomForestRegressor()
random_forest_reg.fit(data_prepared, data_labels)
random_forest_pred = random_forest_reg.predict(data_prepared)
random_forest_rmses = -cross_val_score(random_forest_reg,data_prepared, data_labels, scoring="neg_root_mean_squared_error", cv=10)
print(pd.Series(random_forest_rmses).describe())

count       10.000000
mean     49359.387195
std       2088.042389
min      45944.710614
25%      47866.574623
50%      49416.772391
75%      50645.048604
max      52865.641214
dtype: float64
